# 03 — Full-year one-arm profile for the department desktop

This version records **current RSS and cumulative peak RSS** throughout the
full pipeline, while retaining the same numerical operator and timing
configuration. Run it from a freshly restarted kernel.


In [1]:
import gc
import os
import platform
import resource
import sys

# This notebook benchmarks the CPU implementation.
# These must be set before importing JAX, FEW, or packages that import them.
os.environ["JAX_PLATFORMS"] = "cpu"
os.environ["CUDA_VISIBLE_DEVICES"] = ""

import jax.numpy as jnp

try:
    import pandas as pd
except ImportError:
    pd = None

try:
    import psutil
except ImportError:
    psutil = None

from few.utils.constants import YRSID_SI
from few.waveform import GenerateEMRIWaveform

import json
import platform
import subprocess
import time
from pathlib import Path

import jax
import numpy as np

from lisaorbits import InterpolatedOrbits
from lisaorbits.file import read_orbit_file

import WDM
import WDM_LISAresponse
from WDM.code.time_delay_filters import _time_shift_jax as time_shift_jax


process = psutil.Process(os.getpid()) if psutil is not None else None
memory_stages: dict[str, dict[str, float | int | None]] = {}


def current_rss_bytes():
    """Current resident set size, including native/JAX allocations."""
    return (
        int(process.memory_info().rss)
        if process is not None
        else None
    )


def peak_rss_bytes():
    """Process high-water RSS. Linux reports ru_maxrss in KiB."""
    value = int(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss)
    if sys.platform == "darwin":
        return value
    return value * 1024


def record_memory(stage: str):
    """Record current and cumulative peak RSS at a named pipeline stage."""
    rss = current_rss_bytes()
    peak = peak_rss_bytes()
    snapshot = {
        "rss_bytes": rss,
        "peak_rss_bytes": peak,
        "rss_gib": rss / 1024**3 if rss is not None else None,
        "peak_rss_gib": peak / 1024**3,
    }
    memory_stages[stage] = snapshot
    rss_text = (
        f"{snapshot['rss_gib']:.3f} GiB"
        if snapshot["rss_gib"] is not None
        else "unavailable"
    )
    print(
        f"Memory [{stage}]: "
        f"RSS={rss_text} | "
        f"peak={snapshot['peak_rss_gib']:.3f} GiB"
    )
    return snapshot


def rss_gib():
    rss = current_rss_bytes()
    return rss / 1024**3 if rss is not None else float("nan")


record_memory("after_imports")

NOTEBOOK_DIR = Path.cwd().resolve()
RESPONSE_ROOT = next(
    (
        path for path in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]
        if (path / "examples").exists()
        and ((path / "src" / "WDM_LISAresponse").exists()
             or (path / "WDM_LISAresponse").exists())
    ),
    None,
)
if RESPONSE_ROOT is None:
    # Installed-package fallback.
    RESPONSE_ROOT = Path(WDM_LISAresponse.__file__).resolve().parents[2]

ORBIT_FILE = RESPONSE_ROOT / "examples" / "esa-trailing-orbits.h5"
if not ORBIT_FILE.exists():
    raise FileNotFoundError(
        "Could not find the orbit file at "
        f"{ORBIT_FILE}. The file should remain in WDM_LISAresponse/examples."
    )


def load_lisa_orbits(path: Path):
    """Load the physical LISA orbit file used by the response calculations."""
    orbit_data = read_orbit_file(str(path))
    orbit_times = orbit_data.t0 + orbit_data.dt * np.arange(orbit_data.size)
    orbits = InterpolatedOrbits(
        orbit_times,
        orbit_data.positions,
        orbit_data.velocities,
    )
    return orbit_data, orbits


def git_state(package_file: str) -> dict[str, object]:
    """Return the current Git commit and whether the checkout is dirty."""
    completed = subprocess.run(
        [
            "git",
            "-C",
            str(Path(package_file).resolve().parent),
            "rev-parse",
            "--show-toplevel",
        ],
        capture_output=True,
        text=True,
    )
    if completed.returncode != 0:
        return {"root": None, "commit": None, "dirty": None}

    root = Path(completed.stdout.strip()).resolve()
    commit = subprocess.run(
        ["git", "-C", str(root), "rev-parse", "HEAD"],
        capture_output=True,
        text=True,
        check=True,
    ).stdout.strip()
    dirty = bool(
        subprocess.run(
            ["git", "-C", str(root), "status", "--porcelain"],
            capture_output=True,
            text=True,
            check=True,
        ).stdout.strip()
    )
    return {"root": str(root), "commit": commit, "dirty": dirty}


def json_ready(value):
    """Convert NumPy and JAX objects into JSON-compatible values."""
    if isinstance(value, dict):
        return {str(key): json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(item) for item in value]
    if hasattr(value, "item") and getattr(value, "ndim", None) == 0:
        return value.item()
    if hasattr(value, "tolist"):
        return value.tolist()
    return value


def save_json(filename: str, payload) -> Path:
    """Save a JSON result beside the notebook."""
    path = NOTEBOOK_DIR / filename
    path.write_text(
        json.dumps(json_ready(payload), indent=2),
        encoding="utf-8",
    )
    print("Saved:", path)
    return path


orbit_data, orbits = load_lisa_orbits(ORBIT_FILE)
record_memory("after_orbits")
YEAR_SI = 365.25 * 24.0 * 3600.0
t0 = float(orbit_data.t0 + YEAR_SI)

print("Notebook directory:", NOTEBOOK_DIR)
print("Orbit file:", ORBIT_FILE)
print("Python:", platform.python_version())
print("JAX:", jax.__version__, "| backend:", jax.default_backend())
print("JAX devices:", jax.devices())
if jax.default_backend() != "cpu":
    raise RuntimeError("Notebook 03 must run on the CPU.")
wdm_execution_representation = getattr(
    time_shift_jax,
    "CPU_REAL_EXECUTION_REPRESENTATION",
    None,
)
wdm_cpu_complex_dispatch_active = (
    wdm_execution_representation == "complex"
    and time_shift_jax._array_uses_cpu_backend(jnp.zeros(1))
)
print("WDM_LISAresponse:", WDM_LISAresponse.__file__)
print("WDM_LISAresponse source:", git_state(WDM_LISAresponse.__file__))
print("WDM:", WDM.__file__)
print("WDM source:", git_state(WDM.__file__))
print("WDM CPU real execution:", wdm_execution_representation)
print("WDM CPU complex dispatch active:", wdm_cpu_complex_dispatch_active)
if not wdm_cpu_complex_dispatch_active:
    raise RuntimeError(
        "The imported WDM checkout does not expose the optimized CPU "
        "real-via-complex execution path. Install the patched WDM checkout "
        "and restart the notebook kernel."
    )


from WDM_LISAresponse.code.lisa_response.one_arm_plan import OneArmWDMStaticPlan
from WDM_LISAresponse.code.lisa_response.response_with_plan import (
    _build_wdm_dT_context_from_wdm,
    _build_wdm_dT_grid,
)
from WDM_LISAresponse.code.lisa_response.waveforms import (
    generate_windowed_padded_waveform,
)

def block_result(value):
    """Synchronise every JAX leaf before stopping a timer."""
    for leaf in jax.tree_util.tree_leaves(value):
        if hasattr(leaf, "block_until_ready"):
            leaf.block_until_ready()
    return value


def timed(function):
    started = time.perf_counter()
    value = function()
    block_result(value)
    return value, float(time.perf_counter() - started)


def timing_summary(values):
    values = np.asarray(values, dtype=float)
    q25, q75 = np.percentile(values, [25.0, 75.0])
    return {
        "median_s": float(np.median(values)),
        "q25_s": float(q25),
        "q75_s": float(q75),
        "iqr_s": float(q75 - q25),
        "minimum_s": float(np.min(values)),
        "maximum_s": float(np.max(values)),
        "all_s": [float(value) for value in values],
    }


def benchmark(function, *, warmups: int, repeats: int):
    for _ in range(int(warmups)):
        block_result(function())

    samples = []
    latest = None
    for _ in range(int(repeats)):
        latest, elapsed = timed(function)
        samples.append(elapsed)
    return latest, timing_summary(samples)


def compare_arrays(candidate, reference, *, m_start: int = 0):
    """Return max-absolute and relative-L2 differences."""
    candidate = jax.numpy.asarray(candidate)[..., m_start:]
    reference = jax.numpy.asarray(reference)[..., m_start:]
    difference = candidate - reference

    difference_power = jax.numpy.sum(jax.numpy.abs(difference) ** 2)
    reference_power = jax.numpy.sum(jax.numpy.abs(reference) ** 2)
    max_abs = jax.numpy.max(jax.numpy.abs(difference))
    rel_l2 = jax.numpy.where(
        reference_power > 0,
        jax.numpy.sqrt(difference_power / reference_power),
        jax.numpy.sqrt(difference_power),
    )
    block_result((max_abs, rel_l2))
    return {"max_abs": float(max_abs), "rel_l2": float(rel_l2)}


def compare_link_maps(candidate, reference, *, m_start: int = 0):
    if set(candidate) != set(reference):
        raise ValueError("Directed-link dictionaries contain different keys.")

    metrics = {
        str(link): compare_arrays(
            candidate[link],
            reference[link],
            m_start=m_start,
        )
        for link in reference
    }
    return {
        "per_link": metrics,
        "max_abs": max(item["max_abs"] for item in metrics.values()),
        "rel_l2": max(item["rel_l2"] for item in metrics.values()),
    }


print("Machine:", platform.node())
print("CPU:", platform.processor())
print(
    "Physical cores:",
    psutil.cpu_count(logical=False) if psutil is not None else "unknown",
)


Memory [after_imports]: RSS=0.515 GiB | peak=0.512 GiB
Memory [after_orbits]: RSS=0.518 GiB | peak=0.515 GiB
Notebook directory: /data/tk593/LISA_Response/LISAresponse/WDM_LISAresponse/examples
Orbit file: /data/tk593/LISA_Response/LISAresponse/WDM_LISAresponse/examples/esa-trailing-orbits.h5
Python: 3.12.12
JAX: 0.9.2 | backend: cpu
JAX devices: [CpuDevice(id=0)]
WDM_LISAresponse: /data/tk593/LISA_Response/LISAresponse/WDM_LISAresponse/src/WDM_LISAresponse/__init__.py
WDM_LISAresponse source: {'root': '/data/tk593/LISA_Response/LISAresponse/WDM_LISAresponse', 'commit': '14cb626cad16cfaf715594e51eb4e668431a5ca2', 'dirty': True}
WDM: /data/tk593/LISA_Response/LISAresponse/WDM_GW_wavelets/src/WDM/__init__.py
WDM source: {'root': '/data/tk593/LISA_Response/LISAresponse/WDM_GW_wavelets', 'commit': 'a927f4f1e9fde6e2b9925a008040b6572ef4d35e', 'dirty': False}
WDM CPU real execution: complex
WDM CPU complex dispatch active: True
Machine: calx038.ast.cam.ac.uk
CPU: x86_64
Physical cores: 20


In [2]:
# Full-year production configuration

N = 3_200_000
dt = 10.0
Nf = 100
q_wdm = 4
d_wdm = 4
A_frac = 0.25
L_TRUNC = 25
CALC_M0 = False

ROW_CHUNK_SIZE = 32_640
LAG_BLOCK_SIZE = 17

SHIFT_KWARGS = {
    "assembly_backend": "production",
    "assembly_precision": "complex64",
    "row_chunk_size": ROW_CHUNK_SIZE,
    "lag_block_size": LAG_BLOCK_SIZE,
    "batch_chunk": 6,
    "tl_tp_mode": "interp",
    "tl_tp_interp_points": 32,
    "tl_tp_interp_kind": "linear",
}

theta = np.pi / 2
phi = 0.0

DWT_WARMUPS = 2
DWT_REPEATS = 11
SHIFT_WARMUPS = 2
SHIFT_REPEATS = 11
PIPELINE_WARMUPS = 2
PIPELINE_REPEATS = 9

PARALLEL_SCHEDULES = [
    {"name": "groups_3_workers_2", "group_size": 3, "workers": 2},
    {"name": "groups_2_workers_3", "group_size": 2, "workers": 3},
    {"name": "groups_1_workers_6", "group_size": 1, "workers": 6},
]

SAVE_RESULTS = True
REUSE_CACHE = True


## 1. Local cache and physical inputs

The raw one-year waveform and WDM coefficients are large and may depend on
FEW, WDM and JAX versions. They are cached beside the notebook with
`_local_wdm_` filenames.

Delete the relevant cache files, or set `REUSE_CACHE=False`, after changing
the waveform generator, WDM transform, `calc_m0`, sampling or tiling.

In [3]:

source_state = git_state(WDM_LISAresponse.__file__)
wdm_source_state = git_state(WDM.__file__)

waveform_cache = NOTEBOOK_DIR / "_local_wdm_03_waveform_cache.npz"
wdm_cache = NOTEBOOK_DIR / "_local_wdm_03_wdm_coefficients.npz"
cache_metadata = NOTEBOOK_DIR / "_local_wdm_03_cache_metadata.json"

# `orbit_data`, `orbits` and `t0` were loaded in the setup cell.
td_gen = GenerateEMRIWaveform(
    "FastKerrEccentricEquatorialFlux",
    sum_kwargs={"pad_output": True, "odd_len": True},
    return_list=True,
    force_backend="cpu",
)

model_kwargs = {
    "td_gen": td_gen,
    "m1": 1.0e6,
    "m2": 10.0,
    "a": 0.9,
    "p0": 8.5,
    "e0": 0.3,
    "x0": 1.0,
    "dist": 1.0,
    "theta": theta,
    "phi": phi,
    "qK": np.pi / 3,
    "phiK": np.pi / 3,
    "Phi_phi0": np.pi / 3,
    "Phi_theta0": 0.0,
    "Phi_r0": np.pi / 3,
    "eps": 1.0e-2,
}

if REUSE_CACHE and waveform_cache.exists():
    with np.load(waveform_cache, allow_pickle=False) as cached:
        times_pad = np.asarray(cached["times_pad"])
        h_plus = np.asarray(cached["h_plus"])
        h_cross = np.asarray(cached["h_cross"])
    waveform_generation_s = 0.0
    print("Reused waveform cache:", waveform_cache)
else:
    started = time.perf_counter()
    times_pad, h_plus, h_cross, padding = (
        generate_windowed_padded_waveform(
            waveform_model="emri",
            t0=t0,
            dt=dt,
            N=N,
            model_kwargs=model_kwargs,
            window_function="tukey",
            window_kwargs={"alpha": 0.005, "tukey_factor": 4.0},
            wdm_Nf=Nf,
            return_padding_info=True,
        )
    )
    waveform_generation_s = time.perf_counter() - started
    np.savez(
        waveform_cache,
        times_pad=times_pad,
        h_plus=h_plus,
        h_cross=h_cross,
    )
    print("Created waveform cache:", waveform_cache)

print("Padded samples:", times_pad.size)
print("Waveform generation_s:", f"{waveform_generation_s:.3f}")
record_memory("after_waveform")


Created waveform cache: /data/tk593/LISA_Response/LISAresponse/WDM_LISAresponse/examples/_local_wdm_03_waveform_cache.npz
Padded samples: 3264000
Waveform generation_s: 9.056
Memory [after_waveform]: RSS=6.553 GiB | peak=6.646 GiB


{'rss_bytes': 7036473344,
 'peak_rss_bytes': 7135780864,
 'rss_gib': 6.553226470947266,
 'peak_rss_gib': 6.645713806152344}

In [4]:
# Build the WDM grid and reusable static one-arm plan.

grid_context, grid_s = timed(
    lambda: _build_wdm_dT_grid(
        times_pad,
        Nf=Nf,
        d=d_wdm,
        A_frac=A_frac,
        q=q_wdm,
        calc_m0=CALC_M0,
    )
)
record_memory("after_grid")

one_arm_plan, plan_s = timed(
    lambda: OneArmWDMStaticPlan.build_from_context(
        grid_context,
        orbits,
        L_trunc=L_TRUNC,
        variable_shift_kwargs=SHIFT_KWARGS,
    )
)
record_memory("after_plan")

print("WDM shape:", (grid_context["wdm"].Nt, grid_context["wdm"].Nf))
print("Grid build_s:", f"{grid_s:.3f}")
print("Static plan build_s:", f"{plan_s:.3f}")
print(
    "Persistent interpolation-table memory [MiB]:",
    one_arm_plan.kernel_memory_bytes / 1024**2,
)


Memory [after_grid]: RSS=6.968 GiB | peak=7.288 GiB
Memory [after_plan]: RSS=6.885 GiB | peak=7.288 GiB
WDM shape: (32640, 100)
Grid build_s: 3.655
Static plan build_s: 0.868
Persistent interpolation-table memory [MiB]: 1.6015625


## 2. WDM transforms and reusable local coefficient cache

In [5]:
def sequential_dwt():
    return _build_wdm_dT_context_from_wdm(
        times_pad,
        h_plus,
        h_cross,
        grid_context["wdm"],
        parallel_wdm_transforms=False,
    )


def parallel_dwt():
    return _build_wdm_dT_context_from_wdm(
        times_pad,
        h_plus,
        h_cross,
        grid_context["wdm"],
        parallel_wdm_transforms=True,
    )


sequential_context, sequential_dwt_stats = benchmark(
    sequential_dwt,
    warmups=DWT_WARMUPS,
    repeats=DWT_REPEATS,
)
parallel_context, parallel_dwt_stats = benchmark(
    parallel_dwt,
    warmups=DWT_WARMUPS,
    repeats=DWT_REPEATS,
)
record_memory("after_dwt_benchmarks")

dwt_plus = compare_arrays(
    parallel_context["w_t_plus"],
    sequential_context["w_t_plus"],
    m_start=1,
)
dwt_cross = compare_arrays(
    parallel_context["w_t_cross"],
    sequential_context["w_t_cross"],
    m_start=1,
)

# Preserve the real coefficient dtype used by the cleaned production path.
w_plus_host = np.asarray(parallel_context["w_t_plus"])
w_cross_host = np.asarray(parallel_context["w_t_cross"])

np.savez(
    wdm_cache,
    times_pad=times_pad,
    w_plus=w_plus_host,
    w_cross=w_cross_host,
    Nf=np.asarray(Nf),
    q_wdm=np.asarray(q_wdm),
    d_wdm=np.asarray(d_wdm),
    A_frac=np.asarray(A_frac),
    calc_m0=np.asarray(CALC_M0),
)
cache_metadata.write_text(
    json.dumps(
        json_ready(
            {
                "source": source_state,
                "configuration": {
                    "N": N,
                    "dt": dt,
                    "Nf": Nf,
                    "q_wdm": q_wdm,
                    "d_wdm": d_wdm,
                    "A_frac": A_frac,
                    "calc_m0": CALC_M0,
                },
                "shape": list(w_plus_host.shape),
                "dtype": str(w_plus_host.dtype),
            }
        ),
        indent=2,
    ),
    encoding="utf-8",
)

print("Sequential DWT:", sequential_dwt_stats)
print("Parallel DWT:", parallel_dwt_stats)
print("DWT comparisons:", dwt_plus, dwt_cross)
print("Saved WDM coefficient cache:", wdm_cache)
record_memory("after_wdm_cache")


Memory [after_dwt_benchmarks]: RSS=7.582 GiB | peak=7.884 GiB
Sequential DWT: {'median_s': 0.2297557389974827, 'q25_s': 0.22540360700077144, 'q75_s': 0.2335194844963553, 'iqr_s': 0.008115877495583845, 'minimum_s': 0.22374219199991785, 'maximum_s': 0.23990821100596804, 'all_s': [0.22374219199991785, 0.22477730800164863, 0.23804820900113555, 0.2354942449965165, 0.2271322259985027, 0.23990821100596804, 0.22602990599989425, 0.23154472399619408, 0.22389668500545667, 0.2297557389974827, 0.22991532099695178]}
Parallel DWT: {'median_s': 0.14480801699392032, 'q25_s': 0.14022650600236375, 'q75_s': 0.1460727704979945, 'iqr_s': 0.005846264495630749, 'minimum_s': 0.13603052100370405, 'maximum_s': 0.1509010779991513, 'all_s': [0.1414416949992301, 0.14298006999888457, 0.1509010779991513, 0.14539673500257777, 0.14480801699392032, 0.14846116799890297, 0.1385957340025925, 0.14578312700177776, 0.1390113170054974, 0.13603052100370405, 0.14636241399421124]}
DWT comparisons: {'max_abs': 0.0, 'rel_l2': 0.0} 

{'rss_bytes': 8128741376,
 'peak_rss_bytes': 8465240064,
 'rss_gib': 7.5704803466796875,
 'peak_rss_gib': 7.883869171142578}

## 3. Six-link scheduling sweep


In [6]:
w_plus = jnp.asarray(w_plus_host)
w_cross = jnp.asarray(w_cross_host)
block_result((w_plus, w_cross))
record_memory("after_device_coefficients")

def runtime_sky_shift():
    return one_arm_plan.apply_device(
        w_plus,
        w_cross,
        theta,
        phi,
    )

baseline_links, baseline_shift_stats = benchmark(
    runtime_sky_shift,
    warmups=SHIFT_WARMUPS,
    repeats=SHIFT_REPEATS,
)
record_memory("after_runtime_sky_benchmark")

schedule_rows = [
    {
        "name": "runtime_sky",
        **baseline_shift_stats,
    }
]
best_schedule = schedule_rows[0]

print("Runtime sky + one-arm:", baseline_shift_stats)


Memory [after_device_coefficients]: RSS=7.570 GiB | peak=7.884 GiB
Memory [after_runtime_sky_benchmark]: RSS=7.851 GiB | peak=8.546 GiB
Runtime sky + one-arm: {'median_s': 0.7697346600034507, 'q25_s': 0.7502096539974445, 'q75_s': 0.7770404250040883, 'iqr_s': 0.026830771006643772, 'minimum_s': 0.7327955560031114, 'maximum_s': 0.7942932859950815, 'all_s': [0.7463353779967292, 0.7697346600034507, 0.7811277420041733, 0.772159941996506, 0.7500126289960463, 0.7862656159995822, 0.7504066789988428, 0.7327955560031114, 0.7594314079979085, 0.7729531080040033, 0.7942932859950815]}


## 4. Complete full-year A1


In [7]:
def baseline_a1():
    context = sequential_dwt()
    return one_arm_plan.apply_device(
        context["w_t_plus"],
        context["w_t_cross"],
        theta,
        phi,
    )

def selected_a1():
    context = parallel_dwt()
    return one_arm_plan.apply_device(
        context["w_t_plus"],
        context["w_t_cross"],
        theta,
        phi,
    )

baseline_a1_output, baseline_a1_stats = benchmark(
    baseline_a1,
    warmups=PIPELINE_WARMUPS,
    repeats=PIPELINE_REPEATS,
)
selected_a1_output, selected_a1_stats = benchmark(
    selected_a1,
    warmups=PIPELINE_WARMUPS,
    repeats=PIPELINE_REPEATS,
)

a1_comparison = compare_link_maps(
    selected_a1_output,
    baseline_a1_output,
    m_start=1,
)

print("Sequential-DWT runtime:", baseline_a1_stats)
print("Parallel-DWT runtime:", selected_a1_stats)
print("A1 comparison:", a1_comparison)


Sequential-DWT runtime: {'median_s': 0.9700943149946397, 'q25_s': 0.9615321230012341, 'q75_s': 0.9764069500015466, 'iqr_s': 0.014874827000312507, 'minimum_s': 0.9472989070054609, 'maximum_s': 0.9874000740019255, 'all_s': [0.9700943149946397, 0.9764069500015466, 0.9791300979995867, 0.9615321230012341, 0.9562550030022976, 0.9874000740019255, 0.9472989070054609, 0.9684812119958224, 0.9716351659953943]}
Parallel-DWT runtime: {'median_s': 0.8906846269965172, 'q25_s': 0.8810423300019465, 'q75_s': 0.9320940520046861, 'iqr_s': 0.05105172200273955, 'minimum_s': 0.8735955749943969, 'maximum_s': 0.9492077979957685, 'all_s': [0.9438348659969051, 0.9028555700060679, 0.9320940520046861, 0.8906846269965172, 0.8810423300019465, 0.8832572640021681, 0.8771282460002112, 0.8735955749943969, 0.9492077979957685]}
A1 comparison: {'per_link': {'(2, 1)': {'max_abs': 0.0, 'rel_l2': 0.0}, '(3, 2)': {'max_abs': 0.0, 'rel_l2': 0.0}, '(1, 3)': {'max_abs': 0.0, 'rel_l2': 0.0}, '(3, 1)': {'max_abs': 0.0, 'rel_l2': 0.

In [12]:
record_memory("final")

from dataclasses import fields, is_dataclass


def retained_array_bytes(value):
    """Count unique NumPy/JAX arrays retained by nested dataclass state."""

    seen_objects = set()
    seen_arrays = set()

    def visit(item):
        if item is None:
            return 0

        object_id = id(item)
        if object_id in seen_objects:
            return 0
        seen_objects.add(object_id)

        if isinstance(item, np.ndarray) or (
            hasattr(item, "nbytes")
            and hasattr(item, "shape")
            and hasattr(item, "dtype")
        ):
            if object_id in seen_arrays:
                return 0
            seen_arrays.add(object_id)
            return int(item.nbytes)

        if is_dataclass(item):
            return sum(
                visit(getattr(item, field.name))
                for field in fields(item)
            )

        if isinstance(item, dict):
            return sum(visit(entry) for entry in item.values())

        if isinstance(item, (list, tuple)):
            return sum(visit(entry) for entry in item)

        return 0

    return visit(value)

results = {
    "source": source_state,
    "dependencies": {
        "WDM": {
            "module_file": str(Path(WDM.__file__).resolve()),
            "source": wdm_source_state,
            "cpu_real_execution_representation": (
                wdm_execution_representation
            ),
            "cpu_complex_dispatch_active": (
                wdm_cpu_complex_dispatch_active
            ),
        },
    },
    "machine": {
        "node": platform.node(),
        "processor": platform.processor(),
        "physical_cores": (
            psutil.cpu_count(logical=False)
            if psutil is not None
            else None
        ),
    },
    "configuration": {
        "N": N,
        "dt": dt,
        "Nf": Nf,
        "L_trunc": L_TRUNC,
        "calc_m0": CALC_M0,
        "shift_kwargs": SHIFT_KWARGS,
    },
    "cache_files": {
        "waveform": str(waveform_cache),
        "wdm_coefficients": str(wdm_cache),
        "metadata": str(cache_metadata),
    },
    "setup_s": {
        "waveform_generation": waveform_generation_s,
        "grid": grid_s,
        "one_arm_static_plan": plan_s,
    },
    "dwt": {
        "sequential": sequential_dwt_stats,
        "parallel": parallel_dwt_stats,
        "plus_comparison": dwt_plus,
        "cross_comparison": dwt_cross,
    },
    "shift_schedules": schedule_rows,
    "selected_schedule": best_schedule,
    "complete_a1": {
        "baseline": baseline_a1_stats,
        "selected": selected_a1_stats,
        "comparison": a1_comparison,
    },
    "memory": {
    "one_arm_static_plan_bytes": retained_array_bytes(one_arm_plan),
    "one_arm_kernel_memory_bytes": int(
        one_arm_plan.kernel_memory_bytes
    ),
    "waveform_host_bytes": int(
        np.asarray(times_pad).nbytes
        + np.asarray(h_plus).nbytes
        + np.asarray(h_cross).nbytes
    ),
    "wdm_coefficients_host_bytes": int(
        np.asarray(w_plus_host).nbytes
        + np.asarray(w_cross_host).nbytes
    ),
    "wdm_coefficients_device_bytes": int(
        w_plus.nbytes + w_cross.nbytes
    ),
    "current_rss_gib": rss_gib(),
    "peak_rss_gib": peak_rss_bytes() / 1024**3,
    "stages": memory_stages,
    },
}

print(json.dumps(json_ready(results), indent=2))

memory_rows = [
    {
        "stage": stage,
        "rss_gib": snapshot["rss_gib"],
        "peak_rss_gib": snapshot["peak_rss_gib"],
    }
    for stage, snapshot in memory_stages.items()
]
if pd is not None:
    memory_table = pd.DataFrame(memory_rows)
    display(memory_table)
else:
    for row in memory_rows:
        print(row)


if SAVE_RESULTS:
    save_json("_local_wdm_03_desktop_profile.json", results)
    if pd is not None:
        schedule_path = (
            NOTEBOOK_DIR / "_local_wdm_03_desktop_shift_schedule.csv"
        )
        pd.DataFrame(schedule_rows).to_csv(
            schedule_path,
            index=False,
        )
        print("Saved:", schedule_path)

        memory_path = NOTEBOOK_DIR / "_local_wdm_03_memory_stages.csv"
        pd.DataFrame(memory_rows).to_csv(memory_path, index=False)
        print("Saved:", memory_path)


Memory [final]: RSS=8.239 GiB | peak=8.918 GiB
{
  "source": {
    "root": "/data/tk593/LISA_Response/LISAresponse/WDM_LISAresponse",
    "commit": "14cb626cad16cfaf715594e51eb4e668431a5ca2",
    "dirty": true
  },
  "dependencies": {
    "WDM": {
      "module_file": "/data/tk593/LISA_Response/LISAresponse/WDM_GW_wavelets/src/WDM/__init__.py",
      "source": {
        "root": "/data/tk593/LISA_Response/LISAresponse/WDM_GW_wavelets",
        "commit": "a927f4f1e9fde6e2b9925a008040b6572ef4d35e",
        "dirty": false
      },
      "cpu_real_execution_representation": "complex",
      "cpu_complex_dispatch_active": true
    }
  },
  "machine": {
    "node": "calx038.ast.cam.ac.uk",
    "processor": "x86_64",
    "physical_cores": 20
  },
  "configuration": {
    "N": 3200000,
    "dt": 10.0,
    "Nf": 100,
    "L_trunc": 25,
    "calc_m0": false,
    "shift_kwargs": {
      "assembly_backend": "production",
      "assembly_precision": "complex64",
      "row_chunk_size": 32640,
      

,stage,rss_gib,peak_rss_gib
0,after_imports,0.514938,0.512314
1,after_orbits,0.518028,0.515366
2,after_waveform,6.553226,6.645714
3,after_grid,6.967655,7.287819
4,after_plan,6.884640,7.287819
5,after_dwt_benchmarks,7.582043,7.883869
6,after_wdm_cache,7.570480,7.883869
7,after_device_coefficients,7.570488,7.883869
8,after_runtime_sky_benchmark,7.850792,8.546200
9,final,8.239155,8.918003


Saved: /data/tk593/LISA_Response/LISAresponse/WDM_LISAresponse/examples/_local_wdm_03_desktop_profile.json
Saved: /data/tk593/LISA_Response/LISAresponse/WDM_LISAresponse/examples/_local_wdm_03_desktop_shift_schedule.csv
Saved: /data/tk593/LISA_Response/LISAresponse/WDM_LISAresponse/examples/_local_wdm_03_memory_stages.csv


In [13]:
# Detailed runtime-sky breakdown.
#
# This cell deliberately synchronises after every stage. Therefore, the sum
# of the stage timings may be slightly larger than the normal end-to-end call,
# where JAX can overlap dispatch and avoid intermediate synchronisation.

from WDM.code.time_delay_filters._time_shift_assembly import (
    _assemble_shift_target_batch_dispatch,
)
from WDM_LISAresponse.code.lisa_response.one_arm_plan import (
    _combine_polarizations_device,
)


BREAKDOWN_WARMUPS = 2
BREAKDOWN_REPEATS = 11

shift_context = one_arm_plan.shift_context

real_dtype = (
    jnp.float32
    if shift_context.resolved_assembly_precision == "complex64"
    else jnp.float64
)
complex_dtype = (
    jnp.complex64
    if shift_context.resolved_assembly_precision == "complex64"
    else jnp.complex128
)

w_plus_device = jnp.asarray(w_plus, dtype=real_dtype)
w_cross_device = jnp.asarray(w_cross, dtype=real_dtype)
block_result((w_plus_device, w_cross_device))


def run_runtime_sky_stages():
    """Run one response evaluation and time each major stage."""

    # ---------------------------------------------------------------
    # 1. Sky geometry, antenna patterns, transfer amplitudes and delays
    # ---------------------------------------------------------------
    started = time.perf_counter()

    amplitude_plus, amplitude_cross, time_shift = one_arm_plan.sky_arrays(
        theta,
        phi,
    )

    sky_geometry_s = time.perf_counter() - started

    # ---------------------------------------------------------------
    # 2. Transfer amplitudes to device and combine plus/cross WDM grids
    # ---------------------------------------------------------------
    started = time.perf_counter()

    unshifted = _combine_polarizations_device(
        jnp.asarray(amplitude_plus, dtype=real_dtype),
        jnp.asarray(amplitude_cross, dtype=real_dtype),
        w_plus_device,
        w_cross_device,
    )
    block_result(unshifted)

    polarization_combination_s = time.perf_counter() - started

    # ---------------------------------------------------------------
    # 3. Validate delays and evaluate the precomputed Tl/Tp tables
    # ---------------------------------------------------------------
    started = time.perf_counter()

    # The operator uses -time_shift.
    runtime_delays = shift_context._validate_delays(
        -np.asarray(time_shift)
    )
    delay_device = jnp.asarray(
        runtime_delays,
        dtype=real_dtype,
    )

    Tl_all, Tp_all = shift_context.interpolation_table.evaluate_device(
        delay_device,
        complex_dtype=complex_dtype,
    )
    block_result((Tl_all, Tp_all))

    tl_tp_interpolation_s = time.perf_counter() - started

    # ---------------------------------------------------------------
    # 4. Existing production WDM shift assembly
    # ---------------------------------------------------------------
    started = time.perf_counter()

    ell_all_device = shift_context._device_static_arrays()["ell_all"]

    shifted = _assemble_shift_target_batch_dispatch(
        shift_context.wdm,
        unshifted,
        delay_device,
        ell_all_device,
        shift_context.offset,
        Tl_all,
        Tp_all,
        Cnm=None,
        assembly_backend=shift_context.resolved_assembly_backend,
        assembly_precision=shift_context.resolved_assembly_precision,
        row_chunk_size=shift_context.config.row_chunk_size,
        lag_block_size=shift_context.config.lag_block_size,
        return_device=True,
    )
    block_result(shifted)

    shift_assembly_s = time.perf_counter() - started

    stage_total_s = (
        sky_geometry_s
        + polarization_combination_s
        + tl_tp_interpolation_s
        + shift_assembly_s
    )

    return shifted, {
        "sky_geometry_and_amplitudes_s": sky_geometry_s,
        "polarization_combination_s": polarization_combination_s,
        "tl_tp_interpolation_s": tl_tp_interpolation_s,
        "shift_assembly_s": shift_assembly_s,
        "stage_total_s": stage_total_s,
    }


# Compile and populate relevant device caches.
for _ in range(BREAKDOWN_WARMUPS):
    breakdown_output, _ = run_runtime_sky_stages()
    block_result(breakdown_output)


breakdown_samples = {
    "sky_geometry_and_amplitudes_s": [],
    "polarization_combination_s": [],
    "tl_tp_interpolation_s": [],
    "shift_assembly_s": [],
    "stage_total_s": [],
}

for _ in range(BREAKDOWN_REPEATS):
    breakdown_output, sample = run_runtime_sky_stages()

    for name in breakdown_samples:
        breakdown_samples[name].append(sample[name])


runtime_breakdown = {
    name: timing_summary(values)
    for name, values in breakdown_samples.items()
}

stage_total_median = runtime_breakdown["stage_total_s"]["median_s"]

runtime_breakdown_rows = []
for name in (
    "sky_geometry_and_amplitudes_s",
    "polarization_combination_s",
    "tl_tp_interpolation_s",
    "shift_assembly_s",
):
    median_s = runtime_breakdown[name]["median_s"]

    runtime_breakdown_rows.append(
        {
            "stage": name.removesuffix("_s"),
            "median_s": median_s,
            "fraction_of_synchronised_total": (
                median_s / stage_total_median
                if stage_total_median > 0.0
                else np.nan
            ),
            "minimum_s": runtime_breakdown[name]["minimum_s"],
            "iqr_s": runtime_breakdown[name]["iqr_s"],
        }
    )


# Normal public end-to-end runtime for comparison.
_, end_to_end_stats = benchmark(
    runtime_sky_shift,
    warmups=BREAKDOWN_WARMUPS,
    repeats=BREAKDOWN_REPEATS,
)

print("Synchronised stage total:", runtime_breakdown["stage_total_s"])
print("Normal end-to-end runtime:", end_to_end_stats)

if pd is not None:
    runtime_breakdown_table = pd.DataFrame(runtime_breakdown_rows)
    display(runtime_breakdown_table)
else:
    for row in runtime_breakdown_rows:
        print(row)


# Numerical check against the normal public path.
public_links = runtime_sky_shift()
block_result(public_links)

breakdown_links = {
    link: breakdown_output[index]
    for index, link in enumerate(one_arm_plan.links)
}
block_result(breakdown_links)

breakdown_comparison = compare_link_maps(
    breakdown_links,
    public_links,
    m_start=1,
)

print("Breakdown vs public path:", breakdown_comparison)

Synchronised stage total: {'median_s': 0.7585597219949705, 'q25_s': 0.7483157319984457, 'q75_s': 0.7639636469975812, 'iqr_s': 0.015647914999135537, 'minimum_s': 0.7357455599994864, 'maximum_s': 0.7692726289969869, 'all_s': [0.7585597219949705, 0.7684256089996779, 0.7501272650042665, 0.7357455599994864, 0.7465041989926249, 0.7618537110029138, 0.7650274080006056, 0.7628998859945568, 0.7692726289969869, 0.7580997770055546, 0.746217746003822]}
Normal end-to-end runtime: {'median_s': 0.7610496870038332, 'q25_s': 0.7584936515013396, 'q75_s': 0.7716958415039699, 'iqr_s': 0.013202190002630232, 'minimum_s': 0.7465035499990336, 'maximum_s': 0.7886891820016899, 'all_s': [0.7886891820016899, 0.7677123999965261, 0.7576881620043423, 0.7465035499990336, 0.77286518800247, 0.7610496870038332, 0.7490252850038814, 0.781489925000642, 0.7705264950054698, 0.759299140998337, 0.760690163995605]}


,stage,median_s,fraction_of_synchronised_total,minimum_s,iqr_s
0,sky_geometry_and_amplitudes,0.236713,0.312055,0.233755,0.002218
1,polarization_combination,0.020735,0.027335,0.019290,0.000745
2,tl_tp_interpolation,0.078721,0.103777,0.070025,0.005724
3,shift_assembly,0.423632,0.558468,0.405484,0.020205


Breakdown vs public path: {'per_link': {'(2, 1)': {'max_abs': 0.0, 'rel_l2': 0.0}, '(3, 2)': {'max_abs': 0.0, 'rel_l2': 0.0}, '(1, 3)': {'max_abs': 0.0, 'rel_l2': 0.0}, '(3, 1)': {'max_abs': 0.0, 'rel_l2': 0.0}, '(2, 3)': {'max_abs': 0.0, 'rel_l2': 0.0}, '(1, 2)': {'max_abs': 0.0, 'rel_l2': 0.0}}, 'max_abs': 0.0, 'rel_l2': 0.0}
